### Heater puller データ解析用 Notebook

セル2: ファイル選択, 透過率データの追加

セル3, 4: STFTを用いたカットオフ時刻の確定

セル5, 6: カットオフ時刻に基づくファイバー径時間発展の推定、データ追加

セル7: Excelへの各種データの保存

セル8: 各種グラフの描画・保存

In [ ]:
# === ファイル選択・透過率データ処理 ===
import os
from pathlib import Path
import re
import tkinter as tk
from tkinter import filedialog
import numpy as np
import pandas as pd

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()

# ファイル選択
root = tk.Tk()
root.withdraw()

initial_dir = PROJECT_ROOT / 'data' / 'pull_data'
file_path = filedialog.askopenfilename(
    initialdir=str(initial_dir),
    title="データファイルを選択してください",
    filetypes=(("Text files", "*.txt"), ("All files", "*.*"))
)

root.destroy()

# 保存先の親ディレクトリ
save_base_dir = PROJECT_ROOT / 'analysis_results'

# 初期化
df_processed = None
save_dir = None
timestamp_dir = None
base = None

if not file_path:
    print("ファイルが選択されませんでした。")

else:
    print(f"選択されたファイル: {file_path}")
    base = os.path.basename(file_path)

    # --- ファイル名先頭のタイムスタンプ抽出 ---
    # 例: YYYYMMDD_HHMMSS_xxx.txt → YYYYMMDD_HHMMSS
    m = re.match(r'^(\d{8}_\d{6})', base)

    if m:
        timestamp_dir = m.group(1)
    else:
        timestamp_dir = "others"

    # --- 保存先フォルダ作成 ---
    save_dir = os.path.join(save_base_dir, timestamp_dir)
    os.makedirs(save_dir, exist_ok=True)
    print(f"保存先フォルダ: {save_dir}")

    # --- データ処理 ---
    if not base[:8].isdigit():
        print("ファイル名がYYYYMMDDで始まらないため、処理せずそのまま読み込みます。")
        df_processed = pd.read_csv(file_path, sep="\t", header=None)

    else:
        df = pd.read_csv(file_path, sep="\t", header=None, skiprows=3)

        # 平滑化
        col2 = df[1].to_numpy()
        smoothed = np.array([col2[i:i+25].mean() for i in range(0, len(col2), 25)])
        smoothed_full = np.repeat(smoothed, 25)[:len(col2)]

        # 正規化
        mask = (df[0] >= 0.004) & (df[0] <= 10.000)
        norm_factor = smoothed_full[mask].mean()
        normalized = smoothed_full / norm_factor * 100

        # 列構成変更
        df[3] = df[2]
        df[2] = normalized

        df_processed = df

# 確認用
df_processed.head()

In [ ]:
# === STFTグラフ描画用パラメータ ===
stft_params = {
    "x_min": 0,
    "x_max": 700,
    "y_min": 0,
    "y_max": 100,
    "v_min": -63,
    "v_max": -40,
    "times_to_mark": [457.5, 487.5, 528.5, 602, 619],
}

In [ ]:
# === STFTグラフ描画 ===
import numpy as np
import matplotlib.pyplot as plt
from scipy.signal import spectrogram

# STFT計算パラメータ
fs = 250
nperseg = 800
noverlap = int(nperseg * 0.75)

# STFTプロット関数
def plot_stft_ax(df, ax, stft_params):
    time = df[0].to_numpy()
    signal = df[3].to_numpy()

    # STFT
    f, t, Sxx = spectrogram(
        signal,
        fs=fs,
        window="hann",
        nperseg=nperseg,
        noverlap=noverlap,
        detrend=False,
        scaling="spectrum",
        mode="psd"
    )

    mask = f <= stft_params["y_max"]
    f = f[mask]
    Sxx = Sxx[mask, :]
    Sxx_dB = 10 * np.log10(Sxx + 1e-20)

    # プロット
    extent = [t[0], t[-1], f[0], f[-1]]
    im = ax.imshow(
        Sxx_dB,
        aspect="auto",
        origin="lower",
        extent=extent,
        cmap="viridis",
        vmin=stft_params["v_min"],
        vmax=stft_params["v_max"]
    )

    ax.set_xlabel("Time [s]")
    ax.set_ylabel("Frequency [Hz]")
    ax.set_xlim(stft_params["x_min"], stft_params["x_max"])
    ax.set_ylim(stft_params["y_min"], stft_params["y_max"])

    for tt in stft_params["times_to_mark"]:
        ax.axvline(x=tt, color="w", linestyle="--", linewidth=1, alpha=0.7)

    return im

# STFTプロット
fig, ax = plt.subplots(figsize=(10, 3))
im = plot_stft_ax(df_processed, ax, stft_params)
fig.colorbar(im, ax=ax, label="Power [dB]")

fig.tight_layout()
# fig.savefig(PROJECT_ROOT / "images" / "stft_example.pdf")
plt.show()
plt.close(fig)

In [ ]:
# === ファイバー径時間発展フィッティング用パラメータ ===
v_um_s = 40.0          # one-side stage speed [um/s]
d0_um = 125.0    # fix diameter at t=t0 (effective start), [um]
d_target_um = [0.500, 0.300]  # 目標径 [um]

In [ ]:
# === ファイバー径時間発展フィッティング ===
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# モデル関数: d(t) = d0 * exp(-v (t-t0) / H)
def diameter_model(t_s, d0_um, v_um_s, H_um, t0_s):
    return d0_um * np.exp(-v_um_s * (t_s - t0_s) / H_um)

# フィッティング関数: y = ln(d/d0) = a + m t
def fit_diameter_time_model(t_cutoff_s, d_cutoff_um, v_um_s, d0_um):
    t_cutoff_s = np.asarray(t_cutoff_s, dtype=float)
    d_cutoff_um = np.asarray(d_cutoff_um, dtype=float)

    valid_mask = (
        np.isfinite(t_cutoff_s)
        & np.isfinite(d_cutoff_um)
        & (d_cutoff_um > 0)
    )

    t_cutoff_s = t_cutoff_s[valid_mask]
    d_cutoff_um = d_cutoff_um[valid_mask]

    y = np.log(d_cutoff_um / d0_um)
    A = np.column_stack([np.ones_like(t_cutoff_s), t_cutoff_s])

    (a_fit, m_fit), residuals, rank, s = np.linalg.lstsq(A, y, rcond=None)

    if m_fit >= 0:
        raise ValueError(f"m must be negative, got m={m_fit:.6g}")

    H_fit_um = float(-v_um_s / m_fit)
    t0_fit_s = float(-a_fit / m_fit)

    y_fit = A @ np.array([a_fit, m_fit])
    ss_res = np.sum((y - y_fit) ** 2)
    ss_tot = np.sum((y - np.mean(y)) ** 2)
    r2_log = 1.0 - ss_res / ss_tot if ss_tot > 0 else np.nan

    d_fit_um = diameter_model(t_cutoff_s, d0_um, v_um_s, H_fit_um, t0_fit_s)

    return {
        "v_um_s": float(v_um_s),
        "d0_um": float(d0_um),
        "t_cutoff_s": t_cutoff_s,
        "d_cutoff_um": d_cutoff_um,
        "a_fit": float(a_fit),
        "m_fit": float(m_fit),
        "H_fit_um": H_fit_um,
        "t0_fit_s": t0_fit_s,
        "r2_log": float(r2_log),
        "d_fit_um": d_fit_um,
    }

# ===== 入力 =====
t_cutoff_s = stft_params["times_to_mark"]
d_cutoff_um = [1.875, 1.376, 0.878, 0.440, 0.383]
d_target_um = [0.500, 0.300]

# ===== fit =====
fit_result = fit_diameter_time_model(t_cutoff_s, d_cutoff_um, v_um_s, d0_um)

# ===== 全時間の径を計算して 5 列目へ =====
t_data_s = df_processed[0].to_numpy()
df_processed[4] = diameter_model(
    t_data_s,
    fit_result["d0_um"],
    fit_result["v_um_s"],
    fit_result["H_fit_um"],
    fit_result["t0_fit_s"],
)

# ===== 500 nm, 300 nm に最も近い行を抽出 =====
target_rows = []
for d_target in d_target_um:
    idx = (df_processed[4] - d_target).abs().idxmin()
    target_rows.append({
        "Target [nm]": int(d_target * 1000),
        "Time [s]": float(df_processed.loc[idx, 0]),
        "Tr 852 [%]": float(df_processed.loc[idx, 2]),
        "Diameter [um]": float(df_processed.loc[idx, 4]),
    })

df_targets = pd.DataFrame(target_rows)

# ===== summary 表示 =====
df_summary = pd.DataFrame({
    "parameter": ["H_fit_mm", "t0_fit_s", "r2_log"],
    "value": [
        fit_result["H_fit_um"] / 1000,
        fit_result["t0_fit_s"],
        fit_result["r2_log"],
    ]
})
display(df_summary)
display(df_targets)



# ===== プロット =====
t_plot_s = np.linspace(
    float(np.min(fit_result["t_cutoff_s"])) - 30,
    float(np.max(fit_result["t_cutoff_s"])) + 30,
    500
)
d_plot_um = diameter_model(
    t_plot_s,
    fit_result["d0_um"],
    fit_result["v_um_s"],
    fit_result["H_fit_um"],
    fit_result["t0_fit_s"],
)

fig, ax = plt.subplots(figsize=(6.5, 3.8))
ax.plot(fit_result["t_cutoff_s"], fit_result["d_cutoff_um"], "o", label="data")
ax.plot(t_plot_s, d_plot_um, label=r"fit: $d_0 \exp\left(-\frac{v(t-t_0)}{H}\right)$")
ax.set_xlabel(r"Time $t$ [s]")
ax.set_ylabel("Diameter d [um]")
ax.legend()
ax.grid(True, alpha=0.3)
fig.tight_layout()
plt.show()
plt.close(fig)

In [ ]:
# === Excel 保存 ===
from openpyxl import Workbook
from openpyxl.utils.dataframe import dataframe_to_rows
import os

wb = Workbook()
ws1 = wb.active
ws1.title = "Sheet1"

# =========================
# Sheet1
# =========================
r = 1

base_name = os.path.splitext(os.path.basename(file_path))[0]

ws1.cell(row=r, column=1, value=base_name)
r += 2

ws1.cell(row=r, column=1, value="Item")
ws1.cell(row=r, column=2, value="Value")
r += 1

rows_basic = [
    ("v[um/s]", fit_result["v_um_s"]),
    ("d0 [um]", fit_result["d0_um"]),
    ("H [mm]", fit_result["H_fit_um"] / 1000),
    ("t0 [s]", fit_result["t0_fit_s"]),
    ("R^2", fit_result["r2_log"]),
]

for k, v in rows_basic:
    ws1.cell(row=r, column=1, value=k)
    ws1.cell(row=r, column=2, value=v)
    r += 1

r += 1

ws1.cell(row=r, column=1, value="t_cutoff [s]")
for i, v in enumerate(fit_result["t_cutoff_s"], start=2):
    ws1.cell(row=r, column=i, value=v)
r += 1

ws1.cell(row=r, column=1, value="d_cutoff [um]")
for i, v in enumerate(fit_result["d_cutoff_um"], start=2):
    ws1.cell(row=r, column=i, value=v)
r += 1

ws1.cell(row=r, column=1, value="d_fit [um]")
for i, v in enumerate(fit_result["d_fit_um"], start=2):
    ws1.cell(row=r, column=i, value=v)
r += 2

for row in target_rows:
    ws1.cell(row=r, column=1, value="d [nm]")
    ws1.cell(row=r, column=2, value=row["Target [nm]"])
    r += 1

    ws1.cell(row=r, column=1, value="t_val [s]")
    ws1.cell(row=r, column=2, value=row["Time [s]"])
    r += 1

    ws1.cell(row=r, column=1, value="tr_val [%]")
    ws1.cell(row=r, column=2, value=row["Tr 852 [%]"])
    r += 1

    ws1.cell(row=r, column=1, value="d_val [nm]")
    ws1.cell(row=r, column=2, value=row["Diameter [um]"] * 1000)
    r += 2

# 列幅だけ調整
ws1.column_dimensions["A"].width = 12

# =========================
# Sheet2
# =========================
ws2 = wb.create_sheet("Sheet2")

df_sheet2 = df_processed[[0, 1, 2, 3, 4]].copy()
df_sheet2.columns = ["Time [s]", "Int 852", "Tr 852", "Int 532", "Diameter [um]"]

for row in dataframe_to_rows(df_sheet2, index=False, header=True):
    ws2.append(row)

# =========================
# 保存
# =========================
os.makedirs(save_dir, exist_ok=True)

save_name = f"{base_name}_analysis.xlsx"
save_path = os.path.join(save_dir, save_name)

wb.save(save_path)

print(f"Saved: {save_path}")

In [ ]:
# === 各種グラフの描画・保存 ===
import os
import matplotlib.pyplot as plt

if df_processed is None:
    raise ValueError("df_processed がありません。最初のセルを先に実行してください。")

if save_dir is None:
    raise ValueError("save_dir がありません。最初のセルを先に実行してください。")

# 元データ
time_s = df_processed[0].to_numpy()
tr852 = df_processed[2].to_numpy()
diam_um = df_processed[4].to_numpy()

# 保存先
base_name = os.path.splitext(os.path.basename(file_path))[0]
os.makedirs(save_dir, exist_ok=True)
save_path = os.path.join(save_dir, f"{base_name}_summary_plots.png")

# 図全体
fig, axes = plt.subplots(3, 2, figsize=(18, 14))

# -------------------------
# 1. 時間 vs 852透過率 (97%-)
# -------------------------
ax = axes[0, 0]
ax.plot(time_s, tr852, linewidth=1)
ax.set_xlabel("Time [s]")
ax.set_ylabel("Transmission [%]")
ax.set_title("Time vs Transmission (97%-)")
ax.set_ylim(97, 101)
ax.grid(True, alpha=0.3)

# -------------------------
# 2. 時間 vs 852透過率 (0%-)
# -------------------------
ax = axes[0, 1]
ax.plot(time_s, tr852, linewidth=1)
ax.set_xlabel("Time [s]")
ax.set_ylabel("Transmission [%]")
ax.set_title("Time vs Transmission")
ax.set_ylim(0, 105)
ax.grid(True, alpha=0.3)

# -------------------------
# 3. 時間 vs STFT
# -------------------------
ax = axes[1, 0]
plot_stft_ax(df_processed, ax, stft_params)
ax.set_title("Time vs STFT")

# -------------------------
# 4. 時間 vs ファイバー径
# -------------------------
ax = axes[1, 1]
ax.plot(fit_result["t_cutoff_s"], fit_result["d_cutoff_um"], "o", label="data")
ax.plot(t_plot_s, d_plot_um, label=r"fit: $d_0 \exp\left(-\frac{v(t-t_0)}{H}\right)$")
ax.set_xlabel("Time [s]")
ax.set_ylabel("Fiber diameter [um]")
ax.set_title("Time vs Fiber diameter")
ax.legend()
ax.grid(True, alpha=0.3)

# -------------------------
# 5. ファイバー径 vs 852透過率 (97%-)
# -------------------------
ax = axes[2, 0]
ax.plot(diam_um, tr852, linewidth=1)
ax.axvline(0.500, linestyle="--", linewidth=1.0)
ax.axvline(0.300, linestyle="--", linewidth=1.0)
ax.set_xlabel("Fiber diameter [um]")
ax.set_ylabel("Transmission [%]")
ax.set_title("Fiber diameter vs Transmission (97%-)")
ax.set_xscale("log")
ax.invert_xaxis()
ax.set_ylim(97, 101)
ax.grid(True, alpha=0.3)

# -------------------------
# 6. ファイバー径 vs 852透過率 (0%-)
# -------------------------
ax = axes[2, 1]
ax.plot(diam_um, tr852, linewidth=1)
ax.axvline(0.500, linestyle="--", linewidth=1.0)
ax.axvline(0.300, linestyle="--", linewidth=1.0)
ax.set_xlabel("Fiber diameter [um]")
ax.set_ylabel("Transmission [%]")
ax.set_title("Fiber diameter vs Tr 852")
ax.set_xscale("log")
ax.invert_xaxis()
ax.set_ylim(0, 105)
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(save_path, dpi=300, bbox_inches="tight")
plt.show()
plt.close(fig)

print(f"Saved: {save_path}")